# Assignment 3 - From Validated Files to Business Data Products
Flow: `CSV -> RAW -> CLEAN -> STAGING -> CURATED -> business answers`

| Schema | Question | What it holds |
|---|---|---|
| raw | What did we receive? | Source copy, all TEXT, no changes |
| clean | What trusted entities do we have? | Typed tables with keys |
| staging | What work is needed between? | Step-by-step working tables |
| curated | What final product do we give? | Business tables |

# 01. Business requirements
Full details: `docs/business_requirements.md`

**Finance**
- Question: How is item sales value changing across product categories over time?
- Sales value = `SUM(price)` (no freight)
- Month = `order_purchase_timestamp`
- Exclude `canceled` and `unavailable` orders
- Missing category = `unknown`
- Final grain: one category in one month

**Operations**
- Question: How is delivery performance changing, and where are late deliveries?
- Delivered = status `delivered` and delivery date not empty
- Late = delivery date is after estimated date (same day = on time)
- Late rate = late orders / delivered orders
- Final grain: one customer state in one month

# 02. Inspect existing data

In [ ]:
import os
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW_DIR = ROOT / "data" / "raw"
PROC_DIR = ROOT / "data" / "processed"

FILES = {
    "customers":   "olist_customers_dataset.csv",
    "orders":      "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "products":    "olist_products_dataset.csv",
    "reviews":     "olist_order_reviews_dataset.csv",
}
for name, f in FILES.items():
    print(name, "| raw:", (RAW_DIR / f).exists(), "| processed:", (PROC_DIR / f).exists())

In [ ]:
for name, f in FILES.items():
    df = pd.read_csv(PROC_DIR / f, nrows=3)
    print("---", name, list(df.columns))

# 03. Connect to PostgreSQL and create schemas
- Password is in `.env` (not in the notebook). `.env` is in `.gitignore`.
- Create the database once: `CREATE DATABASE olist_warehouse;`

**Pause and observe - why is a password in the notebook dangerous?**
- Git keeps history forever. Deleting the line later does not remove it from old commits.
- Repos get shared or become public. Bots scan GitHub for passwords.
- People reuse passwords, so one leak can open other systems.
- Notebook outputs can also show the password.
- `.env` keeps secrets outside the code. Each person uses their own.

In [ ]:
load_dotenv(ROOT / ".env")
url = os.getenv("DATABASE_URL")
assert url, "DATABASE_URL missing. Create .env (see .env.example)"
url = url.replace("postgresql://", "postgresql+psycopg://", 1)
engine = create_engine(url)

def run_sql(sql):
    # run one or more statements in one transaction
    with engine.begin() as conn:
        for stmt in sql.split(";"):
            if stmt.strip():
                conn.execute(text(stmt))

def query(sql):
    return pd.read_sql(text(sql), engine)

def value(sql):
    return query(sql).iloc[0, 0]

RESULTS = []
def check(stage, name, ok, evidence=""):
    RESULTS.append({"stage": stage, "check": name, "result": "PASS" if ok else "FAIL", "evidence": evidence})
    print("PASS" if ok else "FAIL", "|", stage, "|", name, "|", evidence)

In [ ]:
run_sql("""
CREATE SCHEMA IF NOT EXISTS raw;
CREATE SCHEMA IF NOT EXISTS clean;
CREATE SCHEMA IF NOT EXISTS staging;
CREATE SCHEMA IF NOT EXISTS curated
""")
query("SELECT schema_name FROM information_schema.schemata WHERE schema_name IN ('raw','clean','staging','curated') ORDER BY 1")

# 04-A. Load RAW
- Source = original files in `data/raw/`
- Every column is TEXT. No fixes. Missing values stay NULL.
- Reloadable: `if_exists="replace"` so a second run does not double the rows.

In [ ]:
import csv

def count_csv_records(path):
    with open(path, newline="", encoding="utf-8") as f:
        return sum(1 for _ in csv.reader(f)) - 1   # minus header

for name, f in FILES.items():
    df = pd.read_csv(RAW_DIR / f, dtype=str)       # dtype=str: no type guessing
    df.to_sql(name, engine, schema="raw", if_exists="replace", index=False, chunksize=10000)
    print("loaded raw." + name, len(df))

### RAW validation (file rows = database rows)

In [ ]:
for name, f in FILES.items():
    file_rows = count_csv_records(RAW_DIR / f)
    db_rows = value(f"SELECT COUNT(*) FROM raw.{name}")
    check("RAW", f"raw.{name} rows match file", file_rows == db_rows, f"file={file_rows} db={db_rows}")

n = value("SELECT COUNT(*) FROM raw.products WHERE product_category_name IS NULL")
check("RAW", "missing category kept as NULL", n > 0, f"NULL categories={n}")
n = value("SELECT COUNT(DISTINCT order_status) FROM raw.orders")
check("RAW", "all order statuses kept", n == 8, f"statuses={n}")

# 04-B. Build CLEAN
| Table | Grain (one row =) | Primary key | Foreign keys |
|---|---|---|---|
| customers | one customer record | customer_id | - |
| products | one product | product_id | - |
| orders | one order | order_id | customer_id -> customers |
| order_items | one item in an order | order_id + order_item_id | order_id -> orders, product_id -> products |
| reviews | one review of an order | review_id + order_id | order_id -> orders |

Load order: customers, products, orders, order_items, reviews (parents first).
Constraints = real rules: price > 0, freight >= 0, score 1 to 5, item number >= 1.

In [ ]:
run_sql("""
DROP TABLE IF EXISTS clean.reviews CASCADE;
DROP TABLE IF EXISTS clean.order_items CASCADE;
DROP TABLE IF EXISTS clean.orders CASCADE;
DROP TABLE IF EXISTS clean.products CASCADE;
DROP TABLE IF EXISTS clean.customers CASCADE;

CREATE TABLE clean.customers (
    customer_id              TEXT PRIMARY KEY,
    customer_unique_id       TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city            TEXT,
    customer_state           CHAR(2) NOT NULL
);

CREATE TABLE clean.products (
    product_id                 TEXT PRIMARY KEY,
    product_category_name      TEXT,
    product_name_lenght        INTEGER,
    product_description_lenght INTEGER,
    product_photos_qty         INTEGER,
    product_weight_g           INTEGER,
    product_length_cm          INTEGER,
    product_height_cm          INTEGER,
    product_width_cm           INTEGER
);

CREATE TABLE clean.orders (
    order_id                      TEXT PRIMARY KEY,
    customer_id                   TEXT NOT NULL REFERENCES clean.customers (customer_id),
    order_status                  TEXT NOT NULL,
    order_purchase_timestamp      TIMESTAMP NOT NULL,
    order_approved_at             TIMESTAMP,
    order_delivered_carrier_date  TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP NOT NULL
);

CREATE TABLE clean.order_items (
    order_id            TEXT NOT NULL REFERENCES clean.orders (order_id),
    order_item_id       INTEGER NOT NULL CHECK (order_item_id >= 1),
    product_id          TEXT NOT NULL REFERENCES clean.products (product_id),
    seller_id           TEXT NOT NULL,
    shipping_limit_date TIMESTAMP,
    price               NUMERIC(10,2) NOT NULL CHECK (price > 0),
    freight_value       NUMERIC(10,2) NOT NULL CHECK (freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);

CREATE TABLE clean.reviews (
    review_id               TEXT NOT NULL,
    order_id                TEXT NOT NULL REFERENCES clean.orders (order_id),
    review_score            SMALLINT NOT NULL CHECK (review_score BETWEEN 1 AND 5),
    review_comment_title    TEXT,
    review_comment_message  TEXT,
    review_creation_date    TIMESTAMP,
    review_answer_timestamp TIMESTAMP,
    PRIMARY KEY (review_id, order_id)
)
""")
print("clean tables created")

### Prepare dataframes with Pandas (types only, no business changes)

In [ ]:
frames = {}
frames["customers"] = pd.read_csv(PROC_DIR / FILES["customers"])

products = pd.read_csv(PROC_DIR / FILES["products"])
for c in ["product_name_lenght","product_description_lenght","product_photos_qty","product_weight_g",
          "product_length_cm","product_height_cm","product_width_cm"]:
    products[c] = products[c].astype("Int64")     # whole numbers that can be empty
frames["products"] = products

frames["orders"] = pd.read_csv(PROC_DIR / FILES["orders"], parse_dates=[
    "order_purchase_timestamp","order_approved_at","order_delivered_carrier_date",
    "order_delivered_customer_date","order_estimated_delivery_date"])

frames["order_items"] = pd.read_csv(PROC_DIR / FILES["order_items"], parse_dates=["shipping_limit_date"])

frames["reviews"] = pd.read_csv(PROC_DIR / FILES["reviews"], parse_dates=["review_creation_date","review_answer_timestamp"])

for k, v in frames.items():
    print(k, v.shape)

In [ ]:
for name in ["customers", "products", "orders", "order_items", "reviews"]:   # parents first
    frames[name].to_sql(name, engine, schema="clean", if_exists="append", index=False, chunksize=10000)
    print("loaded clean." + name, value(f"SELECT COUNT(*) FROM clean.{name}"))

### CLEAN validation

In [ ]:
# CLEAN validation (needs: value, query, check, FILES, frames from earlier cells)

# Guard: stop early if any CLEAN table is empty (load cell was not run)
for t in ["customers", "products", "orders", "order_items", "reviews"]:
    n = int(value(f"SELECT COUNT(*) FROM clean.{t}"))
    assert n > 0, f"clean.{t} is empty. Run the CLEAN load cell first."

# 1. Keys not null and unique
PKS = {
    "customers":   ["customer_id"],
    "products":    ["product_id"],
    "orders":      ["order_id"],
    "order_items": ["order_id", "order_item_id"],
    "reviews":     ["review_id", "order_id"],
}
for t, cols in PKS.items():
    key = ", ".join(cols)
    nulls = sum(int(value(f"SELECT COUNT(*) FROM clean.{t} WHERE {c} IS NULL")) for c in cols)
    dups = int(value(f"SELECT COUNT(*) FROM (SELECT {key} FROM clean.{t} GROUP BY {key} HAVING COUNT(*) > 1) x"))
    check("CLEAN", f"{t}: key has no NULL and no duplicate", nulls == 0 and dups == 0, f"nulls={nulls} dups={dups}")

# 2. No orphan rows (child, parent, join condition, parent key)
RELS = [
    ("orders -> customers",     "clean.orders c",      "clean.customers p", "c.customer_id = p.customer_id", "p.customer_id"),
    ("order_items -> orders",   "clean.order_items c", "clean.orders p",    "c.order_id = p.order_id",       "p.order_id"),
    ("order_items -> products", "clean.order_items c", "clean.products p",  "c.product_id = p.product_id",   "p.product_id"),
    ("reviews -> orders",       "clean.reviews c",     "clean.orders p",    "c.order_id = p.order_id",       "p.order_id"),
]
for label, child, parent, cond, pk in RELS:
    n = int(value(f"SELECT COUNT(*) FROM {child} LEFT JOIN {parent} ON {cond} WHERE {pk} IS NULL"))
    check("CLEAN", f"no orphans: {label}", n == 0, f"orphans={n}")

# 3. Row counts: clean = processed file = raw table
for name in FILES:
    clean_rows = int(value(f"SELECT COUNT(*) FROM clean.{name}"))
    raw_rows = int(value(f"SELECT COUNT(*) FROM raw.{name}"))
    file_rows = len(frames[name])
    check("CLEAN", f"{name}: rows = processed file", clean_rows == file_rows, f"clean={clean_rows} file={file_rows}")
    check("CLEAN", f"{name}: rows = raw table", clean_rows == raw_rows, f"clean={clean_rows} raw={raw_rows}")

# 4. Types: the date columns must be timestamps (named list, no guessing)
DATE_COLS = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
             "order_delivered_customer_date", "order_estimated_delivery_date"]
types = query("SELECT column_name, data_type FROM information_schema.columns "
              "WHERE table_schema = 'clean' AND table_name = 'orders'")
found = types[types.column_name.isin(DATE_COLS)]
bad = found[found.data_type != "timestamp without time zone"].column_name.tolist()
check("CLEAN", "order dates are timestamps",
      len(found) == len(DATE_COLS) and not bad,
      f"found={len(found)}/{len(DATE_COLS)} wrong_type={bad}")

# 5. Totals: money must match (small tolerance for rounding)
db_sum = value("SELECT SUM(price) FROM clean.order_items")
file_total = float(frames["order_items"]["price"].sum())
if db_sum is None or pd.isna(db_sum):
    check("CLEAN", "SUM(price) = processed file", False, "SUM(price) is NULL - clean.order_items is empty. Run the load cell first.")
else:
    db_total = float(db_sum)
    check("CLEAN", "SUM(price) = processed file", abs(db_total - file_total) < 0.01,
          f"db={db_total:,.2f} file={file_total:,.2f}")

# Summary of this cell
clean_checks = [r for r in RESULTS if r["stage"] == "CLEAN"]
print("CLEAN checks:", len(clean_checks), "| failed:", sum(r["result"] == "FAIL" for r in clean_checks))

# 05. Finance pipeline
**Grain:** staging 1 = one order item | staging 2 = one reportable order item | curated = one category in one month

**Prediction:** staging 1 has the same rows as `order_items` (112,650), because each item has exactly one order and one product.

### Staging 1 - `staging.finance_items_enriched`
- One row = one order item
- Why: put item, order and product info in one table
- Validate: rows = order_items, grain unique, no NULL keys

In [ ]:
run_sql("""
DROP TABLE IF EXISTS staging.finance_items_enriched;

CREATE TABLE staging.finance_items_enriched AS
SELECT oi.order_id, oi.order_item_id, oi.product_id,
       o.order_status, o.order_purchase_timestamp,
       oi.price, oi.freight_value, p.product_category_name
FROM clean.order_items oi
JOIN clean.orders o        ON o.order_id = oi.order_id
LEFT JOIN clean.products p ON p.product_id = oi.product_id;

ALTER TABLE staging.finance_items_enriched ADD PRIMARY KEY (order_id, order_item_id)
""")
n1 = value("SELECT COUNT(*) FROM staging.finance_items_enriched")
check("FIN-STG1", "rows = clean.order_items", n1 == value("SELECT COUNT(*) FROM clean.order_items"), f"rows={n1}")
check("FIN-STG1", "no NULL price", value("SELECT COUNT(*) FROM staging.finance_items_enriched WHERE price IS NULL") == 0)
check("FIN-STG1", "SUM(price) same as clean",
      value("SELECT SUM(price) FROM staging.finance_items_enriched") == value("SELECT SUM(price) FROM clean.order_items"))
query("SELECT * FROM staging.finance_items_enriched LIMIT 5")

### Staging 2 - `staging.finance_items_reportable`
- One row = one order item that Finance can report
- Why: apply Finance rules (remove canceled/unavailable, month, `unknown` category)
- Prediction: fewer rows than staging 1 (549 removed)

In [ ]:
run_sql("""
DROP TABLE IF EXISTS staging.finance_items_reportable;

CREATE TABLE staging.finance_items_reportable AS
SELECT order_id, order_item_id, order_status,
       CAST(date_trunc('month', order_purchase_timestamp) AS date)  AS reporting_month,
       COALESCE(NULLIF(TRIM(product_category_name), ''), 'unknown') AS product_category,
       price AS item_sales_value
FROM staging.finance_items_enriched
WHERE order_status NOT IN ('canceled', 'unavailable');

ALTER TABLE staging.finance_items_reportable ADD PRIMARY KEY (order_id, order_item_id)
""")
expected = value("SELECT COUNT(*) FROM clean.order_items oi JOIN clean.orders o USING (order_id) WHERE o.order_status NOT IN ('canceled','unavailable')")
n2 = value("SELECT COUNT(*) FROM staging.finance_items_reportable")
check("FIN-STG2", "rows = eligible items from clean", n2 == expected, f"expected={expected} actual={n2} removed={n1-n2}")
check("FIN-STG2", "no NULL month / category",
      value("SELECT COUNT(*) FROM staging.finance_items_reportable WHERE reporting_month IS NULL OR product_category IS NULL") == 0)
print("items with unknown category:", value("SELECT COUNT(*) FROM staging.finance_items_reportable WHERE product_category = 'unknown'"))
query("SELECT * FROM staging.finance_items_reportable LIMIT 5")

### Curated - `curated.finance_monthly_category_sales`
- One row = one product category in one month
- `item_sales_value` = SUM(price), no freight
- `distinct_orders` = COUNT(DISTINCT order_id). Do **not** add it across categories.

In [ ]:
run_sql("""
DROP TABLE IF EXISTS curated.finance_monthly_category_sales;

CREATE TABLE curated.finance_monthly_category_sales (
    reporting_month  DATE          NOT NULL,
    product_category TEXT          NOT NULL,
    item_sales_value NUMERIC(14,2) NOT NULL,
    items_sold       INTEGER       NOT NULL,
    distinct_orders  INTEGER       NOT NULL,
    PRIMARY KEY (reporting_month, product_category)
);

INSERT INTO curated.finance_monthly_category_sales
SELECT reporting_month, product_category, SUM(item_sales_value), COUNT(*), COUNT(DISTINCT order_id)
FROM staging.finance_items_reportable
GROUP BY reporting_month, product_category
""")

### Finance reconciliation (curated vs CLEAN, same rules)

In [ ]:
clean_total = value("SELECT SUM(oi.price) FROM clean.order_items oi JOIN clean.orders o USING (order_id) WHERE o.order_status NOT IN ('canceled','unavailable')")
cur_total = value("SELECT SUM(item_sales_value) FROM curated.finance_monthly_category_sales")
check("FIN-CUR", "total sales = CLEAN", cur_total == clean_total, f"clean={clean_total} curated={cur_total}")

check("FIN-CUR", "items_sold = eligible items", value("SELECT SUM(items_sold) FROM curated.finance_monthly_category_sales") == expected)

pairs = value("SELECT COUNT(*) FROM (SELECT DISTINCT reporting_month, product_category FROM staging.finance_items_reportable) x")
check("FIN-CUR", "rows = distinct month + category", value("SELECT COUNT(*) FROM curated.finance_monthly_category_sales") == pairs, f"rows={pairs}")

diff = value("""
SELECT COUNT(*) FROM curated.finance_monthly_category_sales f
JOIN (
  SELECT CAST(date_trunc('month', o.order_purchase_timestamp) AS date) m,
         COALESCE(NULLIF(TRIM(p.product_category_name), ''), 'unknown') c,
         SUM(oi.price) s, COUNT(DISTINCT oi.order_id) d
  FROM clean.order_items oi
  JOIN clean.orders o USING (order_id)
  LEFT JOIN clean.products p USING (product_id)
  WHERE o.order_status NOT IN ('canceled','unavailable')
  GROUP BY 1, 2
) r ON r.m = f.reporting_month AND r.c = f.product_category
WHERE r.s <> f.item_sales_value OR r.d <> f.distinct_orders
""")
check("FIN-CUR", "every row matches CLEAN recalculation", diff == 0, f"different rows={diff}")

### Finance business query

In [ ]:
query("""
SELECT reporting_month, product_category, item_sales_value, distinct_orders
FROM curated.finance_monthly_category_sales
ORDER BY reporting_month, item_sales_value DESC
LIMIT 20
""")

# 06. Operations pipeline
**Grain:** staging 1 = one order | staging 2 = one order with a delivery class | curated = one state in one month

**Prediction:** staging 1 has 99,441 rows (same as orders). Each order has one customer, so the join does not add rows.

### Staging 1 - `staging.ops_orders_base`
- One row = one order
- Why: collect the evidence to judge delivery (dates, status, state)

In [ ]:
run_sql("""
DROP TABLE IF EXISTS staging.ops_orders_base;

CREATE TABLE staging.ops_orders_base AS
SELECT o.order_id, o.customer_id, c.customer_state, c.customer_city,
       o.order_status, o.order_purchase_timestamp,
       o.order_estimated_delivery_date, o.order_delivered_customer_date
FROM clean.orders o
JOIN clean.customers c ON c.customer_id = o.customer_id;

ALTER TABLE staging.ops_orders_base ADD PRIMARY KEY (order_id)
""")
check("OPS-STG1", "rows = clean.orders", value("SELECT COUNT(*) FROM staging.ops_orders_base") == value("SELECT COUNT(*) FROM clean.orders"))
check("OPS-STG1", "no NULL state / status / purchase date / estimate",
      value("SELECT COUNT(*) FROM staging.ops_orders_base WHERE customer_state IS NULL OR order_status IS NULL OR order_purchase_timestamp IS NULL OR order_estimated_delivery_date IS NULL") == 0)
query("SELECT * FROM staging.ops_orders_base LIMIT 5")

### Staging 2 - `staging.ops_orders_classified`
- One row = one order with a delivery class
- Why: put the late rule in one place, so inspect single orders before the rate

| Case | Class |
|---|---|
| canceled / unavailable | not_fulfilled |
| delivered, no delivery date | unknown |
| delivered date > estimated date | late |
| delivered date <= estimated date | on_time |
| other status | not_delivered |

In [ ]:
run_sql("""
DROP TABLE IF EXISTS staging.ops_orders_classified;

CREATE TABLE staging.ops_orders_classified AS
SELECT order_id, customer_state, customer_city, order_status,
       CAST(date_trunc('month', order_purchase_timestamp) AS date) AS reporting_month,
       order_estimated_delivery_date, order_delivered_customer_date,
       CASE WHEN order_status = 'delivered' AND order_delivered_customer_date IS NOT NULL
            THEN CAST(order_delivered_customer_date AS date) - CAST(order_estimated_delivery_date AS date)
       END AS days_vs_estimate,
       CASE
         WHEN order_status IN ('canceled','unavailable')                                    THEN 'not_fulfilled'
         WHEN order_status = 'delivered' AND order_delivered_customer_date IS NULL          THEN 'unknown'
         WHEN order_status = 'delivered'
          AND CAST(order_delivered_customer_date AS date) > CAST(order_estimated_delivery_date AS date) THEN 'late'
         WHEN order_status = 'delivered'                                                    THEN 'on_time'
         ELSE 'not_delivered'
       END AS delivery_classification
FROM staging.ops_orders_base;

ALTER TABLE staging.ops_orders_classified ADD PRIMARY KEY (order_id)
""")
check("OPS-STG2", "rows = staging 1", value("SELECT COUNT(*) FROM staging.ops_orders_classified") == value("SELECT COUNT(*) FROM staging.ops_orders_base"))
check("OPS-STG2", "every order has a class", value("SELECT COUNT(*) FROM staging.ops_orders_classified WHERE delivery_classification IS NULL") == 0)
check("OPS-STG2", "late > 0 days, on_time <= 0 days",
      value("SELECT COUNT(*) FROM staging.ops_orders_classified WHERE (delivery_classification='late' AND days_vs_estimate <= 0) OR (delivery_classification='on_time' AND days_vs_estimate > 0)") == 0)
check("OPS-STG2", "canceled orders are never late",
      value("SELECT COUNT(*) FROM staging.ops_orders_classified WHERE order_status IN ('canceled','unavailable') AND delivery_classification='late'") == 0)
query("SELECT delivery_classification, COUNT(*) AS orders FROM staging.ops_orders_classified GROUP BY 1 ORDER BY 2 DESC")

### Curated - `curated.operations_monthly_state_delivery`
- One row = one customer state in one month
- `late_delivery_rate` = late orders / delivered orders (NULL when delivered = 0)

In [ ]:
run_sql("""
DROP TABLE IF EXISTS curated.operations_monthly_state_delivery;

CREATE TABLE curated.operations_monthly_state_delivery (
    reporting_month    DATE    NOT NULL,
    customer_state     CHAR(2) NOT NULL,
    total_orders       INTEGER NOT NULL,
    delivered_orders   INTEGER NOT NULL,
    late_orders        INTEGER NOT NULL,
    late_delivery_rate NUMERIC(6,4),
    PRIMARY KEY (reporting_month, customer_state)
);

INSERT INTO curated.operations_monthly_state_delivery
WITH counts AS (
  SELECT reporting_month, customer_state,
         COUNT(*) AS total_orders,
         COUNT(*) FILTER (WHERE delivery_classification IN ('late','on_time')) AS delivered_orders,
         COUNT(*) FILTER (WHERE delivery_classification = 'late')              AS late_orders
  FROM staging.ops_orders_classified
  GROUP BY reporting_month, customer_state
)
SELECT reporting_month, customer_state, total_orders, delivered_orders, late_orders,
       ROUND(CAST(late_orders AS numeric) / NULLIF(delivered_orders, 0), 4)
FROM counts
""")

### Operations reconciliation

In [ ]:
check("OPS-CUR", "total orders = clean.orders", value("SELECT SUM(total_orders) FROM curated.operations_monthly_state_delivery") == value("SELECT COUNT(*) FROM clean.orders"))

clean_delivered = value("SELECT COUNT(*) FROM clean.orders WHERE order_status='delivered' AND order_delivered_customer_date IS NOT NULL")
check("OPS-CUR", "delivered = CLEAN", value("SELECT SUM(delivered_orders) FROM curated.operations_monthly_state_delivery") == clean_delivered, f"delivered={clean_delivered}")

clean_late = value("SELECT COUNT(*) FROM clean.orders WHERE order_status='delivered' AND CAST(order_delivered_customer_date AS date) > CAST(order_estimated_delivery_date AS date)")
check("OPS-CUR", "late = CLEAN", value("SELECT SUM(late_orders) FROM curated.operations_monthly_state_delivery") == clean_late, f"late={clean_late}")

check("OPS-CUR", "late <= delivered <= total on every row",
      value("SELECT COUNT(*) FROM curated.operations_monthly_state_delivery WHERE late_orders > delivered_orders OR delivered_orders > total_orders") == 0)
check("OPS-CUR", "rate between 0 and 1 (or NULL)",
      value("SELECT COUNT(*) FROM curated.operations_monthly_state_delivery WHERE late_delivery_rate < 0 OR late_delivery_rate > 1") == 0)

### Operations business query - where are late deliveries?

In [ ]:
query("""
SELECT customer_state, SUM(delivered_orders) AS delivered, SUM(late_orders) AS late,
       ROUND(SUM(late_orders)::numeric / NULLIF(SUM(delivered_orders), 0), 4) AS late_rate
FROM curated.operations_monthly_state_delivery
GROUP BY customer_state
ORDER BY late_rate DESC
LIMIT 10
""")

In [ ]:
query("""
SELECT reporting_month, SUM(delivered_orders) AS delivered, SUM(late_orders) AS late,
       ROUND(SUM(late_orders)::numeric / NULLIF(SUM(delivered_orders), 0), 4) AS late_rate
FROM curated.operations_monthly_state_delivery
GROUP BY reporting_month
ORDER BY reporting_month
""")

# Optional - dim_date
One row per calendar date. Not required by the assignment. Good for month / quarter / weekday columns.

In [ ]:
run_sql("""
DROP TABLE IF EXISTS clean.dim_date;

CREATE TABLE clean.dim_date AS
SELECT d::date AS date_key,
       EXTRACT(YEAR FROM d)::int AS year,
       EXTRACT(QUARTER FROM d)::int AS quarter,
       EXTRACT(MONTH FROM d)::int AS month,
       TO_CHAR(d, 'FMMonth') AS month_name,
       date_trunc('month', d)::date AS month_start,
       TO_CHAR(d, 'FMDay') AS day_name,
       EXTRACT(ISODOW FROM d) IN (6, 7) AS is_weekend
FROM generate_series('2016-01-01'::date, '2018-12-31'::date, interval '1 day') AS d;

ALTER TABLE clean.dim_date ADD PRIMARY KEY (date_key)
""")
missing = value("SELECT COUNT(*) FROM clean.orders o LEFT JOIN clean.dim_date d ON d.date_key = CAST(o.order_purchase_timestamp AS date) WHERE d.date_key IS NULL")
check("DIM", "every purchase date exists in dim_date", missing == 0, f"missing={missing}")

# 07. Final validation and reflection

In [ ]:
evidence = pd.DataFrame(RESULTS)
print("Checks:", len(evidence), "| Passed:", (evidence.result == "PASS").sum(), "| Failed:", (evidence.result == "FAIL").sum())
evidence[evidence.result == "FAIL"]

In [ ]:
query("""
SELECT table_schema AS layer, table_name FROM information_schema.tables
WHERE table_schema IN ('raw','clean','staging','curated') ORDER BY 1, 2
""")

In [ ]:
# Save the evidence table to docs/ (optional)
lines = ["# Validation Evidence", "", f"Checks: {len(evidence)} | Passed: {(evidence.result=='PASS').sum()} | Failed: {(evidence.result=='FAIL').sum()}", "",
         "| Stage | Check | Result | Evidence |", "|---|---|---|---|"]
for r in evidence.itertuples():
    lines.append(f"| {r.stage} | {r.check} | {r.result} | {r.evidence} |")
(ROOT / "docs" / "validation_evidence_generated.md").write_text("\n".join(lines), encoding="utf-8")

### What I learned
- Know the grain before joining.
- Predict row counts, then check them.
- Staging steps are easier to debug than one big query.
- Curated tables make the business question simple.
- Validate at every stage.

### What I would improve
- Add a `dim_date` table.
- Add indexes if tables grow.
- Move the SQL into a script when automation is allowed.